# mdkeychunker quickstart: chunk Markdown, then check where your RAG budget should go

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bhavik-mangla/MDKeyChunker/blob/main/examples/diagnose_quickstart.ipynb)

This notebook shows the two things the package is for:

1. **`mk.chunk`**: header-aware chunks with the section path in the text you embed. No dependencies, no LLM.
2. **`mk.diagnose`**: on your own documents and a few labelled questions, test whether a reranker, a stronger
   embedder or LLM enrichment would find more of the right text, with honest "inconclusive" verdicts when you
   don't have enough questions.

Background: *[Upgrade the Retriever First: What an LLM Call per Chunk Buys in RAG](https://arxiv.org/abs/2603.23533)*.
On Colab, use a GPU runtime (Runtime → Change runtime type → T4) for part 2.

In [ ]:
!pip install -q "mdkeychunker[retrieval,tokens]"
import mdkeychunker as mk
print(mk.__version__)

## 1. Chunk a Markdown document

Each chunk keeps its raw `text`, the `embed_text` you index (section path + text), the header stack,
character offsets into the source and a content-stable id.

In [ ]:
doc = """# Billing guide

## Invoices

Invoices are sent on the first day of each month. You can download past invoices from Settings > Billing.

## Refunds

Refunds for annual plans take five business days. Monthly plans are refunded as account credit.

## API tokens

Rotate the API token from Settings > API. Old tokens stop working after one hour.
"""
chunks = mk.chunk(doc, doc_id="billing", max_tokens=60)
for c in chunks:
    print(c.chunk_id, c.section_path, (c.start_char, c.end_char), c.token_count)
    print("  embed_text:", repr(c.embed_text[:70]))

## 2. Diagnose a retrieval stack on real data

We use 10 research papers from Qasper (CC BY 4.0) bundled with the package's test suite: 39 questions, each with
the evidence paragraph annotators marked. `diagnose` builds BM25, a dense retriever, their hybrid, a stronger
embedder and a reranker, runs every question through each, and compares them on the same questions.

With only 39 questions, expect **"inconclusive"** verdicts. That is the point: the report tells you how large a
difference these questions could detect, so you know whether to label more before spending money. On 60 papers
(198 questions) the same check found a clear reranker gain (+11.5 points) and no detectable gain from LLM enrichment.

In [ ]:
import ast, json, urllib.request
URL = "https://raw.githubusercontent.com/bhavik-mangla/MDKeyChunker/main/benchmarks/ci_bench/qasper_ci.json"
papers = json.load(urllib.request.urlopen(URL))["papers"]
gold_of = lambda q: ast.literal_eval(q["evidence"]) if isinstance(q["evidence"], str) else q["evidence"]

chunks = [c for p in papers for c in mk.chunk(p["markdown"], doc_id=p["id"])]
qa = [{"question": q["question"], "doc_id": p["id"], "gold": gold_of(q)}
      for p in papers for q in p["questions"]]
print(len(papers), "papers,", len(chunks), "chunks,", len(qa), "questions")

In [ ]:
report = mk.diagnose(chunks, qa,
                     embedder="qwen3-embedding-0.6b",          # the dense half of the hybrid
                     upgrade_embedder=None,                     # set a stronger model to test an upgrade
                     reranker="qwen3-reranker-0.6b",
                     embed_batch_size=8, max_seq_length=2048)   # keep memory modest on small GPUs
from IPython.display import Markdown
Markdown(report.to_markdown())

## 3. Your own data

You need 30-200 questions, each with **the text that answers it** (copied from your documents, not a chunk id),
so the same labels work for any chunker.

```python
qa = [
    {"question": "How long do annual refunds take?", "doc_id": "billing",
     "gold": ["Refunds for annual plans take five business days"]},
    # ...
]
```

Bring your own search (your vector DB, Elasticsearch, pgvector ...) as a function returning chunk ids or chunks,
and compare it with the built-in arms:

```python
def my_search(query, k):
    return [hit.id for hit in my_index.search(query, top_k=k)]   # ids must match your chunks' chunk_id

report = mk.diagnose(chunks, qa, retrievers={"mine": my_search}, scope="corpus")
```

Chunks from LangChain, LlamaIndex or Chonkie work too: `mk.diagnose(langchain_docs, qa)`.
Testing LLM enrichment: pass `enriched=mk.enrich(chunks, llm)` and see `mk.estimate_cost(...)` for the bill.

## 4. Compare chunkers on the same questions (CPU, seconds)

Because the gold is text, you can score any chunking with `mk.evidence_recall`. Here: header-led chunks vs
256-token windows under BM25, searched within each paper.

In [ ]:
from mdkeychunker.retrieval import BM25Retriever

def mean_recall(make_chunks):
    scores = []
    for p in papers:
        cs = make_chunks(p)
        r = BM25Retriever(cs, method="study")
        for q in p["questions"]:
            gold = gold_of(q)
            scores.append(mk.evidence_recall(r.search(q["question"], k=20), gold, budget_tokens=512))
    return round(100 * sum(scores) / len(scores), 1)

print("header-led chunks:", mean_recall(lambda p: mk.chunk(p["markdown"], doc_id=p["id"])))
print("256-token windows:", mean_recall(lambda p: mk.window(p["markdown"], size=256, overlap=32, doc_id=p["id"])))

With 39 questions these numbers are noisy, and this is BM25 only. On this sample windows may even come out
ahead, while on all 951 Qasper questions under the study's hybrid retriever header-led chunks beat both
512-character and 256-token windows by 7.5 points (and 256-token windows won on FreshStack code docs). Small
samples mislead: score both chunkings on enough of your own questions, with confidence intervals, before deciding.

**Next:** [README](https://github.com/bhavik-mangla/MDKeyChunker) · [paper](https://arxiv.org/abs/2603.23533) ·
[study data](https://huggingface.co/datasets/bhavikmangla/mdkeychunker-study)